In [1]:
import numpy as np
import random
from google.colab import files
import pandas as pd
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows

def calcular_capacitancia_equivalente(matriz):
    soma_paralelo = np.sum(matriz, axis=0)
    inverso_serie = np.sum(1.0 / soma_paralelo)
    return 1.0 / inverso_serie

def trocar_capacitancias(matrizes):
    idx1, idx2 = random.sample(range(len(matrizes)), 2)
    pos1 = (random.randrange(n_lin), random.randrange(n_col))
    pos2 = (random.randrange(n_lin), random.randrange(n_col))
    matrizes[idx1][pos1], matrizes[idx2][pos2] = matrizes[idx2][pos2], matrizes[idx1][pos1]

def diferenca_total_capacitancias(capacitancias):
    return np.ptp(capacitancias)

# Parâmetros da matriz de capacitores
n_col = 4
n_lin = 4

# Número total de capacitâncias
num_capacitancias = 2 * 3 * n_col * n_lin

# Gerar capacitâncias aleatórias
capacitancias = np.random.uniform(15e-6, 25e-6, num_capacitancias)
nr_serie      = np.random.uniform(1000, 10000, num_capacitancias)

# Criar as matrizes de capacitores
capacitancias_por_matriz = n_col * n_lin
matrizes = []
for i in range(0, num_capacitancias, capacitancias_por_matriz):
    matriz = capacitancias[i:i + capacitancias_por_matriz].reshape(n_lin, n_col)
    matrizes.append(matriz)

# Calculando as capacitâncias equivalentes iniciais
capacitancias_equivalentes_iniciais = []
for matriz in matrizes:
    cap_equiv = calcular_capacitancia_equivalente(matriz)
    capacitancias_equivalentes_iniciais.append(cap_equiv)

matrizes_originais = matrizes
# Parâmetros iniciais da otimização
n_iteracoes = 10000
menor_diferenca = float('inf')
melhor_configuracao = None

# Loop de otimização
for _ in range(n_iteracoes):
    trocar_capacitancias(matrizes)
    capacitancias_equivalentes_atualizadas = []
    for matriz in matrizes:
        cap_equiv = calcular_capacitancia_equivalente(matriz)
        capacitancias_equivalentes_atualizadas.append(cap_equiv)

    diferenca_atual = diferenca_total_capacitancias(capacitancias_equivalentes_atualizadas)
    if diferenca_atual < menor_diferenca:
        menor_diferenca = diferenca_atual
        melhor_configuracao = [np.copy(matriz) for matriz in matrizes]

# Restaurando as matrizes para a melhor configuração encontrada
matrizes = melhor_configuracao
capacitancias_equivalentes = []
for matriz in matrizes:
    cap_equiv = calcular_capacitancia_equivalente(matriz)
    capacitancias_equivalentes.append(cap_equiv)

# Exibindo os resultados
print("Capacitâncias equivalentes antes da otimização:")
for cap_equiv in capacitancias_equivalentes_iniciais:
    print(f"{cap_equiv:.2e} F")

print("\nCapacitâncias equivalentes após a otimização:")
for cap_equiv in capacitancias_equivalentes:
    print(f"{cap_equiv:.2e} F")


Capacitâncias equivalentes antes da otimização:
1.94e-05 F
1.99e-05 F
1.94e-05 F
2.14e-05 F
2.17e-05 F
1.86e-05 F

Capacitâncias equivalentes após a otimização:
2.01e-05 F
2.01e-05 F
2.00e-05 F
2.01e-05 F
1.99e-05 F
2.01e-05 F


In [ ]:


# Transformando as matrizes em DataFrames
dataframes_iniciais = [pd.DataFrame(matriz) for matriz in matrizes_originais]
dataframes_otimizados = [pd.DataFrame(matriz) for matriz in melhor_configuracao]

# Criando um arquivo Excel para as matrizes iniciais
excel_file_iniciais = 'matrizes_capacitancias_iniciais.xlsx'
with pd.ExcelWriter(excel_file_iniciais, engine='openpyxl') as writer:
    for i, df_inicial in enumerate(dataframes_iniciais):
        df_inicial.to_excel(writer, sheet_name=f'Matriz_{i}', index=True)

# Criando um arquivo Excel para as matrizes otimizadas
excel_file_otimizadas = 'matrizes_capacitancias_otimizadas.xlsx'
with pd.ExcelWriter(excel_file_otimizadas, engine='openpyxl') as writer:
    for i, df_otimizado in enumerate(dataframes_otimizados):
        df_otimizado.to_excel(writer, sheet_name=f'Matriz_{i}', index=True)


files.download(excel_file_iniciais)
files.download(excel_file_otimizadas)

print(f"Matrizes iniciais foram salvas em '{excel_file_iniciais}'.")
print(f"Matrizes otimizadas foram salvas em '{excel_file_otimizadas}'.")



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Matrizes iniciais foram salvas em 'matrizes_capacitancias_iniciais.xlsx'.
Matrizes otimizadas foram salvas em 'matrizes_capacitancias_otimizadas.xlsx'.
